# 07. Context Engineering
## 검색된 문서에서 필요한 근거만 LLM에 전달하기

### 이번 실습의 위치

```text
05 Retrieval Engineering
        ↓
06 Reranking
        ↓
관련 문서 Top-k
        ↓
07 Context Engineering
        ↓
필요한 근거만 Context로 구성
        ↓
08 Grounded Answer
```

이번 실습의 핵심은 하나이다.

> **검색된 문서 전체를 LLM에 넣는 것이 아니라, 질문에 필요한 근거만 Context로 만든다.**

### 학습 목표

1. Retrieval과 Context Engineering의 차이를 설명한다.
2. 검색 문서 전체를 그대로 넣을 때의 문제를 이해한다.
3. 질문과 관련된 근거만 추출해 Compact Context를 만든다.
4. Context가 짧고 명확할수록 답변 생성에 유리할 수 있음을 이해한다.


# 1. Retrieval과 Context Engineering의 차이

Retrieval은 **관련 문서를 찾는 단계**이다.

Context Engineering은 **찾은 문서 중 무엇을 LLM에게 전달할지 결정하는 단계**이다.

```text
Retrieval
질문과 관련된 문서를 찾는다.
        ↓
Top-k Documents
        ↓
Context Engineering
질문에 필요한 내용만 남긴다.
        ↓
Compact Context
        ↓
LLM
```

예를 들어 검색된 문서가 3개라고 해도,
각 문서의 모든 문장이 질문에 필요한 것은 아니다.


# 2. 환경 설정

실습을 단순하게 유지하기 위해 이번 파일에서는 복잡한 Hybrid/Reranking 코드를 다시 구현하지 않는다.

강의 흐름에서는 **06번에서 선택된 상위 문서**에 Context Engineering을 적용한다고 이해하면 된다.

이 노트북은 독립적으로 실행할 수 있도록 ChromaDB에서 상위 문서 3개를 가져와 같은 개념을 실습한다.


In [1]:
# 상위 폴더의 common_config.py를 불러오기 위한 경로 설정
import sys
from pathlib import Path

from langchain_chroma import Chroma

sys.path.append("..")
from common_config import embedding_model, llm_connect


# 02번에서 만든 ChromaDB 위치
DB_PATH = Path("chroma_db/ai_trend_report_doc_chunks")
COLLECTION_NAME = "ai_trend_report_2026_doc_chunks"

if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_docling_chunks.ipynb를 먼저 실행하세요."
    )

# 임베딩 모델과 LLM 준비
embeddings = embedding_model()
llm = llm_connect(
    temperature=0,
    max_tokens=500,
)

# 기존 ChromaDB 연결
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)

print("저장된 chunk 수:", vector_store._collection.count())


저장된 chunk 수: 48


# 3. 검색된 문서 확인

먼저 질문과 관련된 문서 3개를 가져온다.

```text
Question
   ↓
Retriever
   ↓
Top-3 Documents
```

여기서 중요한 점은 **관련 문서를 찾았다고 해서 문서 전체를 LLM에 모두 넣을 필요는 없다는 것**이다.


In [2]:
# 예제 질문
question = "기업 내부에서 AI 에이전트는 어떤 방식으로 활용될 것으로 전망되나요?"

# 관련 문서 3개 검색
docs = vector_store.similarity_search(
    question,
    k=3,
)

# 어떤 문서가 검색되었는지 확인
for rank, doc in enumerate(docs, start=1):
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


1. chunk=38 | pages=17
2. chunk=27 | pages=13
3. chunk=6 | pages=4


# 4. Before: 문서 전체를 Context로 사용

가장 단순한 RAG는 검색된 문서를 그대로 이어 붙여 LLM에 전달할 수 있다.

```text
Doc 1 전체
+
Doc 2 전체
+
Doc 3 전체
        ↓
      LLM
```

하지만 이렇게 하면 질문과 관계없는 내용까지 함께 들어갈 수 있다.

Context Engineering은 이 Context를 **질문 중심으로 정리하는 과정**이다.


In [3]:
# 검색된 문서 3개의 전체 내용을 하나의 Context로 합친다.
before_context = ""

for i, doc in enumerate(docs, start=1):
    before_context += (
        f"\n[문서 {i} | pages={doc.metadata.get('pages')}]\n"
        f"{doc.page_content}\n"
    )

print("[Before Context]")
print(before_context[:2000])

# 복잡한 토큰 계산 대신 문자열 길이로 크기 차이만 간단히 확인한다.
print("\nBefore Context 길이:", len(before_context))


[Before Context]

[문서 1 | pages=17]
4.2 2026년 AI 산업 전망
- 2026년 AI 시장은 선도 기업을 중심으로 내재화가 빠르게 진행되며 시장 규모가 지속 확대될 전망
- Ÿ AI 활용이 여러 산업군에서 실험 단계를 넘어 일부 핵심 공정과 서비스 운영에 본격 도입되기 시작하면서 AI가 기업 운영 인프라로 자리 잡아가는 흐름이 확산될 가능성이 높음
- Ÿ 특히 대기업·디지털 강도가 높은 산업을 중심으로 의사결정·운영·고객 대응 등 주요 영역에 AI가 단계적으로 통합되며 AI 관련 시장 수요가 꾸준한 성장세를 이어갈 것으로 평가됨
- AI 활용이 확산되면서 고품질·도메인 특화 데이터(버티컬 데이터)의 중요성이 크게
- 높아지고 있으며, 데이터 확보·관리·유통을 둘러싼 생태계가 빠르게 성장하는 중임
- Ÿ 이와 함께 기존 산업의 가치 사슬도 전면 재편이라기보다는 핵심 공정·고부가 서비스 단에서
- 데이터·AI 중심 구조로 이동하는 압력이 강화되는 양상을 보일 것으로 전망됨
- 기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며
- 사람-에이전트-시스템이 혼합된 업무 구조가 일부 영역에서 확산될 가능성이 있음
- Ÿ 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는 수준으로 평가됨

[문서 2 | pages=13]
3.2 토픽 모델링 결과
3.2.1 [AI 산업 토픽] - 시장 규모 확대와 비즈니스 구조 전환 중심
- AI 산업 워드클라우드는 도입확산, 시장 규모, 인프라 전환이라는 세 축을 중심으로 재편되는 결과를 보임
- 도입, 확대, 성장세, 확산: AI 도입·확산 가속에 따른 산업 구조 변화
- Ÿ AI 활용이 시범 적용(PoC) 단계를 넘어 전사적·범용적 도입 단계로 확산되고 있음을 보여줌
- Ÿ 업종별 도입 범위가 커지며 산업 구조와 경쟁 환경이 재

# 5. 질문에 필요한 근거만 추출하기

이번 실습에서는 Context Engineering을 가장 단순한 형태로 구현한다.

```text
Question
   +
검색 문서
   ↓
LLM
   ↓
질문에 직접 필요한 근거만 추출
   ↓
Compact Context
```

LLM에게 다음 원칙만 준다.

1. 질문과 직접 관련된 내용만 남긴다.
2. 문서에 없는 내용을 추가하지 않는다.
3. 핵심 근거만 간결하게 정리한다.
4. 근거가 어느 문서에서 왔는지 표시한다.

복잡한 threshold, tokenizer, token budget 코드는 이번 실습에서는 다루지 않는다.


In [4]:
# 검색된 문서를 하나의 텍스트로 준비한다.
documents_text = ""

for i, doc in enumerate(docs, start=1):
    documents_text += f"""
[문서 {i}]
{doc.page_content}

"""


# LLM에게 "답변"을 만들라고 하지 않고
# 질문에 필요한 "근거"만 추출하도록 요청한다.
prompt = f"""
다음 문서에서 질문에 답하는 데 직접 필요한 근거만 추출하세요.

규칙:
- 질문과 관련된 내용만 남기세요.
- 문서에 없는 내용을 추가하지 마세요.
- 핵심 근거를 5개 이내로 정리하세요.
- 각 근거 앞에 [문서 번호]를 표시하세요.
- 최종 답변은 작성하지 마세요.

질문:
{question}

문서:
{documents_text}
"""

response = llm.invoke(prompt)

# 추출된 근거가 Compact Context가 된다.
compact_context = response.content

print("[Compact Context]")
print(compact_context)


[Compact Context]
[문서 1] 기업 내부에서 AI 에이전트를 활용한 **문서 처리, 고객 지원, 운영 자동화**가 증가할 것으로 언급됨.  
[문서 1] **사람-에이전트-시스템이 혼합된 업무 구조**가 일부 영역에서 확산될 가능성이 있다고 제시됨.  
[문서 2] **에이전트 도입 확산**이 업무 흐름과 운영 방식의 **재구성 초기 징후**로 해석됨.  
[문서 2] AI가 **인프라·에이전트 기반 운영 모델**로 변화하고 있다는 점이 언급됨.


# 6. Before vs After 비교

이제 Context Engineering 적용 전후를 비교한다.

```text
Before
검색 문서 전체
        ↓
긴 Context


After
검색 문서
   ↓
질문에 필요한 근거만 추출
   ↓
Compact Context
```

여기서는 정교한 token 계산보다 **Context가 얼마나 간결해졌는지**만 확인한다.


In [5]:
print("Before Context 길이 :", len(before_context))
print("Compact Context 길이:", len(compact_context))

print("\n줄어든 문자 수:")
print(len(before_context) - len(compact_context))


Before Context 길이 : 1897
Compact Context 길이: 241

줄어든 문자 수:
1656


# 7. 실제 서비스에서는 무엇을 더 할 수 있는가?

이번 실습에서는 이해를 위해 **Evidence Extraction 하나만 직접 구현**했다.

실제 Context Engineering에서는 다음 방법도 사용할 수 있다.

```text
Context Selection
→ 어떤 문서를 넣을 것인가?

Evidence Extraction
→ 문서에서 어떤 근거를 넣을 것인가?

Token Budget
→ Context를 얼마나 넣을 것인가?
```

하지만 지금 단계에서는 세 가지를 모두 코드로 구현할 필요는 없다.

핵심은 다음 질문이다.

> **“검색된 내용을 그대로 넣을 것인가, 질문에 필요한 내용만 넣을 것인가?”**


# 8. 핵심 정리

## 오늘 기억할 것

```text
Retriever / Reranker
        ↓
관련 문서
        ↓
Context Engineering
        ↓
질문에 필요한 근거만 선택·압축
        ↓
Compact Context
        ↓
LLM
```

### 한 문장으로 정리

> **Context Engineering은 검색된 문서를 더 많이 넣는 기술이 아니라, LLM이 답변에 필요한 근거를 더 잘 볼 수 있도록 Context를 구성하는 과정이다.**

---

## 다음 실습

```text
08 Grounded Answer
```

07번에서 만든 Compact Context를 이용해  
08번에서는 **근거 기반 답변과 출처 표시**로 넘어간다.
